In [ ]:
# OmniVoice's own lockfile, from PyTorch's cu128 index for torch; torchvision at
# the version that belongs to that torch so transformers can import beside it.
!pip -q install "torch==2.8.0" "torchaudio==2.8.0" "torchvision==0.23.0" --index-url https://download.pytorch.org/whl/cu128 2>&1 | tail -3
!pip -q install "transformers==5.3.0" "accelerate==1.13.0" "webdataset==1.0.2" "safetensors==0.7.0" "tokenizers==0.22.2" "huggingface-hub==1.7.2" "tensorboardX==2.6.4" "soundfile==0.13.1" "librosa==0.11.0" "pydub==0.25.1" 2>&1 | tail -3
!pip -q install --no-deps "git+https://github.com/k2-fsa/OmniVoice@08be0b4ccbac3e13e374e86fbfead4b4cac343e2" 2>&1 | tail -3
!pip -q install speechbrain faster-whisper jiwer pyloudnorm noisereduce num2words silero-vad onnxruntime 2>&1 | tail -3


In [ ]:
import hashlib
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path


def captured(command):
    result = subprocess.run(command, capture_output=True, text=True)
    return result.returncode, (result.stdout + result.stderr).strip()


probe = subprocess.run([sys.executable, "-c", "from transformers import PreTrainedModel"],
                       text=True, capture_output=True)
if probe.returncode != 0 and "torchvision" in probe.stderr:
    print("transformers cannot import beside this torchvision - removing torchvision")
    print(captured([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision"])[1])
    probe = subprocess.run([sys.executable, "-c", "from transformers import PreTrainedModel"],
                           text=True, capture_output=True)
assert probe.returncode == 0, probe.stderr[-2000:]

import torch

OFFICIAL = {'torch': '2.8.0+cu128', 'torchaudio': '2.8.0+cu128', 'transformers': '5.3.0', 'accelerate': '1.13.0', 'webdataset': '1.0.2', 'safetensors': '0.7.0', 'tokenizers': '0.22.2', 'huggingface-hub': '1.7.2', 'tensorboardX': '2.6.4', 'soundfile': '0.13.1', 'librosa': '0.11.0', 'pydub': '0.25.1', 'numpy': '2.4.3', 'triton': '3.4.0'}
PINNED = ('torch', 'torchaudio', 'transformers', 'accelerate', 'webdataset', 'safetensors', 'tokenizers', 'huggingface-hub')


def installed(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def matches(have, want):
    if have is None:
        return False
    return have == want or (have.split("+")[0] == want.split("+")[0] and "+" not in want)


stack = {name: {"official": want, "installed": installed(name),
                "matches": matches(installed(name), want), "pinned_here": name in PINNED}
         for name, want in OFFICIAL.items()}
direct = json.loads(importlib.metadata.distribution("omnivoice").read_text("direct_url.json") or "{}")
code, driver = captured(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"])
ENVIRONMENT_RECORD = {
    "python": platform.python_version(),
    "torch_cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "bf16_supported_including_emulation": torch.cuda.is_bf16_supported(),
    "gpus": [{"name": torch.cuda.get_device_name(i), "capability": list(torch.cuda.get_device_capability(i))}
             for i in range(torch.cuda.device_count())],
    "nvidia_smi": driver,
    "official_stack": stack,
    "omnivoice": {"version": installed("omnivoice"),
                  "commit": (direct.get("vcs_info") or {}).get("commit_id"), "url": direct.get("url")},
    "other_packages": {name: installed(name) for name in ("torchvision", "ipykernel", "papermill", "scipy")},
}
Path("/kaggle/working/environment.json").write_text(json.dumps(ENVIRONMENT_RECORD, indent=2))
for name, row in stack.items():
    mark = "ok" if row["matches"] else ("DIFFERS" if row["pinned_here"] else "differs - image's version kept")
    print(f"{name:16} official {row['official']:12} installed {str(row['installed']):16} {mark}")
print("omnivoice", ENVIRONMENT_RECORD["omnivoice"])
print("cuda", torch.version.cuda, "| cudnn", torch.backends.cudnn.version(), "|", driver)
for name in PINNED:
    assert stack[name]["matches"], f"{name} is not the official {OFFICIAL[name]}: {stack[name]['installed']}"
assert ENVIRONMENT_RECORD["omnivoice"]["commit"] == "08be0b4ccbac3e13e374e86fbfead4b4cac343e2", ENVIRONMENT_RECORD["omnivoice"]
assert torch.cuda.device_count() >= 2, "the official recipe trains on two GPUs - use T4 x2"
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"cuda:{i}: {torch.cuda.get_device_name(i)}, capability {torch.cuda.get_device_capability(i)}, "
          f"{free / 2**30:.1f} GiB free of {total / 2**30:.1f}")
print("disk:", f"{shutil.disk_usage('/tmp').free / 2**30:.0f} GiB free under /tmp")

INPUT = Path("/kaggle/input")
found = sorted(p for p in INPUT.rglob("repo/scripts/render_frontend_conditions.py") if "persian-voice-frontend-listening-v2" in p.parts)
if not found:
    for path in sorted(p for p in INPUT.rglob("*") if p.is_dir())[:40]:
        print("   ", path)
    raise SystemExit("no repo with scripts/render_frontend_conditions.py under /kaggle/input")
REPO = Path("/kaggle/working/repo")
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(found[0].parent.parent, REPO)
# Kaggle unpacks the .gz when it builds the dataset; either form is checked
# against the hash the split builder recorded for its decompressed content.
MANIFESTS = REPO / "eval/manifests"
SPLITS = next((p for p in (MANIFESTS / "adaptation_splits_v1.jsonl.gz", MANIFESTS / "adaptation_splits_v1.jsonl")
               if p.is_file()), None)
assert SPLITS is not None, "the dataset predates the adaptation splits"
import gzip  # noqa: E402

content = gzip.open(SPLITS).read() if SPLITS.suffix == ".gz" else SPLITS.read_bytes()
expected = json.loads((MANIFESTS / "adaptation_splits_v1.summary.json").read_text())["records_sha256"]
assert hashlib.sha256(content).hexdigest() == expected, "the split file is not the one the builder wrote"
print("splits:", SPLITS.name, "sha256 matches the summary")

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONPATH"] = str(REPO / "src")
os.environ["VOICECLONE_SCORING_ENV"] = "kaggle:persian-voice-news-vowels-run-v1"
sys.path.insert(0, str(REPO / "scripts"))
from kernel_gates import Gates, report_group, run_bounded  # noqa: E402

GATES = Gates()
RUNS = {}
LOGS = Path("/kaggle/working/logs")
LOGS.mkdir(exist_ok=True)
WORK = Path("/tmp/adapt")
WORK.mkdir(parents=True, exist_ok=True)
RECORDS = Path("/kaggle/working/records")
RECORDS.mkdir(exist_ok=True)


In [ ]:
from runner_check import check, diagnostics  # noqa: E402

ok, reason, run = check(run_bounded, Path("/kaggle/working/runner_check"), REPO / "scripts/moss15_worker.py")
RUNS["runner_check"] = run.record()
print(run.summary())
seen, line = diagnostics(report_group)
print("descriptor report:", "works" if seen else "DOES NOT WORK", "-", line)
GATES.record("runner", ok, reason)


In [ ]:
def fetch(plan, minutes, flags=()):
    """scripts/fetch_inputs.py under run_bounded: every request timed out, the stage bounded."""
    plan_path, report_path = RECORDS / "fetch_plan.json", RECORDS / "fetch.json"
    plan_path.write_text(json.dumps(plan))
    run = run_bounded([sys.executable, str(REPO / "scripts/fetch_inputs.py"), "--plan", str(plan_path),
                       "--report", str(report_path), *flags], timeout=minutes * 60, log=LOGS / "fetch.log",
                      monitor_every=300)
    RUNS["fetch"] = run.record()
    print(run.summary())
    report = json.loads(report_path.read_text()) if report_path.is_file() else {}
    fetched = {step["name"]: {k: step.get(k) for k in ("gib", "seconds", "tries", "failed_check")}
               for step in report.get("steps", [])}
    return run.ok and report.get("ok") is True, fetched

(REPO / "eval/runs/news-vowels-v1/P/texts.json").parent.mkdir(parents=True, exist_ok=True)
(REPO / "eval/runs/news-vowels-v1/P/texts.json").write_text('{\n "sentences_file": "eval/manifests/news_vowels_v1.json",\n "sentences_sha256": "5b3fe83fd6f943d0b5319f2a2bb164a754497ece40fa30af665f9bb134566c69",\n "code_commit": "fe7aae9",\n "conditions": {\n  "PB": "the published weights reading the app\'s text",\n  "PV": "the published weights, every first vowel"\n },\n "sentences": [\n  {\n   "id": "n01",\n   "category": "news",\n   "text": "شرکت سامسونگ اعلام کرد گوشی\u200cهای تاشوی جدیدش را از دوشنبه دوازدهم اکتبر در فروشگاه\u200cهای اروپا عرضه می\u200cکند.",\n   "marked_in_V": [\n    "شرکت",\n    "اعلام",\n    "جدیدش",\n    "دوازدهم",\n    "اکتبر",\n    "فروشگاه\u200cهای",\n    "اروپا"\n   ],\n   "app_notes": [\n    "Ezafe written in on 5 word(s) by hazm\'s ParsBERT tagger: شرکت، گوشی\u200cهای، تاشوی، دوازدهم، فروشگاه\u200cهای.",\n    "First vowel written on 2 homograph(s) from Homo-GE2PE: کرد، عرضه."\n   ],\n   "texts": {\n    "PB": "شرکتِ سامسونگ اعلام کَرد گوشی\u200cهایِ تاشویِ جدیدش را از دوشنبه دوازدهمِ اکتبر در فروشگاه\u200cهایِ اروپا عَرضه می\u200cکند.",\n    "PV": "شِرکتِ سامسونگ اِعلام کَرد گوشی\u200cهایِ تاشویِ جَدیدش را از دوشنبه دَوازدهمِ اُکتبر در فُروشگاه\u200cهایِ اُروپا عَرضه می\u200cکند."\n   }\n  },\n  {\n   "id": "n02",\n   "category": "news",\n   "text": "به گفته مدیرعامل این کمپانی، کارخانه تازه مجهز به ربات\u200cهای صنعتی است و تا پایان نوامبر راه\u200cاندازی می\u200cشود.",\n   "marked_in_V": [\n    "گفته",\n    "مدیرعامل",\n    "کمپانی",\n    "مجهز",\n    "ربات\u200cهای",\n    "صنعتی",\n    "است",\n    "نوامبر"\n   ],\n   "app_notes": [\n    "Ezafe written in on 5 word(s) by hazm\'s ParsBERT tagger: گفته، مدیرعامل، کارخانه، ربات\u200cهای، پایان."\n   ],\n   "texts": {\n    "PB": "به گفتهٔ مدیرعاملِ این کمپانی، کارخانهٔ تازه مجهز به ربات\u200cهایِ صنعتی است و تا پایانِ نوامبر راه\u200cاندازی می\u200cشود.",\n    "PV": "به گُفتهٔ مُدیرعاملِ این کُمپانی، کارخانهٔ تازه مُجهز به رُبات\u200cهایِ صَنعتی اَست و تا پایانِ نُوامبر راه\u200cاندازی می\u200cشود."\n   }\n  },\n  {\n   "id": "n03",\n   "category": "news",\n   "text": "تیم ملی فوتبال جمعه شب در ورزشگاه آزادی با نتیجه دو بر یک به حریف عراقی خود باخت.",\n   "marked_in_V": [\n    "ملی",\n    "ورزشگاه",\n    "نتیجه",\n    "حریف",\n    "عراقی"\n   ],\n   "app_notes": [\n    "Ezafe written in on 6 word(s) by hazm\'s ParsBERT tagger: تیم، ملی، ورزشگاه، نتیجه، حریف، عراقی.",\n    "First vowel written on 1 homograph(s) from Homo-GE2PE: بر."\n   ],\n   "texts": {\n    "PB": "تیمِ ملیِ فوتبال جمعه شب در ورزشگاهِ آزادی با نتیجهٔ دو بَر یک به حریفِ عراقیِ خود باخت.",\n    "PV": "تیمِ مِلیِ فوتبال جمعه شب در وَرزشگاهِ آزادی با نَتیجهٔ دو بَر یک به حَریفِ عِراقیِ خود باخت."\n   }\n  },\n  {\n   "id": "n04",\n   "category": "news",\n   "text": "سازمان هواشناسی پیش\u200cبینی کرده است که از سه\u200cشنبه هفته آینده دمای هوا در استان\u200cهای شمالی کاهش می\u200cیابد.",\n   "marked_in_V": [\n    "هواشناسی",\n    "کرده",\n    "است",\n    "سه\u200cشنبه",\n    "هفته",\n    "دمای",\n    "هوا",\n    "استان\u200cهای",\n    "شمالی"\n   ],\n   "app_notes": [\n    "Ezafe written in on 5 word(s) by hazm\'s ParsBERT tagger: سازمان، سه\u200cشنبه، هفته، دمای، استان\u200cهای."\n   ],\n   "texts": {\n    "PB": "سازمانِ هواشناسی پیش\u200cبینی کرده است که از سه\u200cشنبهٔ هفتهٔ آینده دمایِ هوا در استان\u200cهایِ شمالی کاهش می\u200cیابد.",\n    "PV": "سازمانِ هَواشناسی پیش\u200cبینی کَرده اَست که از سِه\u200cشنبهٔ هَفتهٔ آینده دَمایِ هَوا در اُستان\u200cهایِ شُمالی کاهش می\u200cیابد."\n   }\n  },\n  {\n   "id": "n05",\n   "category": "news",\n   "text": "پژوهشگران دانشگاه تهران موفق شدند دارویی برای درمان سرطان خون بسازند که هزینه کمتری دارد.",\n   "marked_in_V": [\n    "پژوهشگران",\n    "تهران",\n    "موفق",\n    "شدند",\n    "برای",\n    "درمان",\n    "سرطان",\n    "بسازند",\n    "هزینه",\n    "کمتری"\n   ],\n   "app_notes": [\n    "Ezafe written in on 6 word(s) by hazm\'s ParsBERT tagger: پژوهشگران، دانشگاه، برای، درمان، سرطان، هزینه."\n   ],\n   "texts": {\n    "PB": "پژوهشگرانِ دانشگاهِ تهران موفق شدند دارویی برایِ درمانِ سرطانِ خون بسازند که هزینهٔ کمتری دارد.",\n    "PV": "پَژوهشگرانِ دانشگاهِ تِهران مُوفق شُدند دارویی بَرایِ دَرمانِ سَرطانِ خون بِسازند که هَزینهٔ کَمتری دارد."\n   }\n  },\n  {\n   "id": "n06",\n   "category": "news",\n   "text": "قیمت دلار در بازار آزاد روز یکشنبه پایین آمد و معامله\u200cگران منتظر تصمیم بانک مرکزی هستند.",\n   "marked_in_V": [\n    "قیمت",\n    "دلار",\n    "یکشنبه",\n    "معامله\u200cگران",\n    "منتظر",\n    "تصمیم",\n    "مرکزی",\n    "هستند"\n   ],\n   "app_notes": [\n    "Ezafe written in on 6 word(s) by hazm\'s ParsBERT tagger: قیمت، بازار، روز، منتظر، تصمیم، بانک."\n   ],\n   "texts": {\n    "PB": "قیمتِ دلار در بازارِ آزاد روزِ یکشنبه پایین آمد و معامله\u200cگران منتظرِ تصمیمِ بانکِ مرکزی هستند.",\n    "PV": "قِیمتِ دُلار در بازارِ آزاد روزِ یِکشنبه پایین آمد و مُعامله\u200cگران مُنتظرِ تَصمیمِ بانکِ مَرکزی هَستند."\n   }\n  },\n  {\n   "id": "n07",\n   "category": "news",\n   "text": "جشنواره فیلم فجر امسال در بهمن ماه برگزار می\u200cشود و هشتاد فیلم کوتاه و بلند در آن حضور دارند.",\n   "marked_in_V": [\n    "جشنواره",\n    "فجر",\n    "امسال",\n    "برگزار",\n    "هشتاد",\n    "بلند",\n    "حضور"\n   ],\n   "app_notes": [\n    "Ezafe written in on 4 word(s) by hazm\'s ParsBERT tagger: جشنواره، فیلم، فجر، فیلم."\n   ],\n   "texts": {\n    "PB": "جشنوارهٔ فیلمِ فجرِ امسال در بهمن ماه برگزار می\u200cشود و هشتاد فیلمِ کوتاه و بلند در آن حضور دارند.",\n    "PV": "جَشنوارهٔ فیلمِ فَجرِ اِمسال در بهمن ماه بَرگزار می\u200cشود و هَشتاد فیلمِ کوتاه و بُلند در آن حُضور دارند."\n   }\n  },\n  {\n   "id": "n08",\n   "category": "news",\n   "text": "وزارت نیرو از مردم خواست در ساعت\u200cهای اوج مصرف، از روشن کردن کولرها و وسایل پرمصرف خودداری کنند.",\n   "marked_in_V": [\n    "وزارت",\n    "مصرف",\n    "کردن",\n    "وسایل",\n    "پرمصرف",\n    "کنند"\n   ],\n   "app_notes": [\n    "Ezafe written in on 5 word(s) by hazm\'s ParsBERT tagger: وزارت، ساعت\u200cهای، اوج، کردن، وسایل.",\n    "First vowel written on 1 homograph(s) from Homo-GE2PE: مردم."\n   ],\n   "texts": {\n    "PB": "وزارتِ نیرو از مَردم خواست در ساعت\u200cهایِ اوجِ مصرف، از روشن کردنِ کولرها و وسایلِ پرمصرف خودداری کنند.",\n    "PV": "وِزارتِ نیرو از مَردم خواست در ساعت\u200cهایِ اوجِ مَصرف، از روشن کَردنِ کولرها و وَسایلِ پُرمصرف خودداری کُنند."\n   }\n  }\n ]\n}\n', encoding="utf-8")
(REPO / "eval/runs/news-vowels-v1/A/texts.json").parent.mkdir(parents=True, exist_ok=True)
(REPO / "eval/runs/news-vowels-v1/A/texts.json").write_text('{\n "sentences_file": "eval/manifests/news_vowels_v1.json",\n "sentences_sha256": "5b3fe83fd6f943d0b5319f2a2bb164a754497ece40fa30af665f9bb134566c69",\n "code_commit": "fe7aae9",\n "conditions": {\n  "AB": "the adapted weights reading the app\'s text",\n  "AV": "the adapted weights, every first vowel"\n },\n "sentences": [\n  {\n   "id": "n01",\n   "category": "news",\n   "text": "شرکت سامسونگ اعلام کرد گوشی\u200cهای تاشوی جدیدش را از دوشنبه دوازدهم اکتبر در فروشگاه\u200cهای اروپا عرضه می\u200cکند.",\n   "marked_in_V": [\n    "شرکت",\n    "اعلام",\n    "جدیدش",\n    "دوازدهم",\n    "اکتبر",\n    "فروشگاه\u200cهای",\n    "اروپا"\n   ],\n   "app_notes": [\n    "Ezafe written in on 5 word(s) by hazm\'s ParsBERT tagger: شرکت، گوشی\u200cهای، تاشوی، دوازدهم، فروشگاه\u200cهای.",\n    "First vowel written on 2 homograph(s) from Homo-GE2PE: کرد، عرضه."\n   ],\n   "texts": {\n    "AB": "شرکتِ سامسونگ اعلام کَرد گوشی\u200cهایِ تاشویِ جدیدش را از دوشنبه دوازدهمِ اکتبر در فروشگاه\u200cهایِ اروپا عَرضه می\u200cکند.",\n    "AV": "شِرکتِ سامسونگ اِعلام کَرد گوشی\u200cهایِ تاشویِ جَدیدش را از دوشنبه دَوازدهمِ اُکتبر در فُروشگاه\u200cهایِ اُروپا عَرضه می\u200cکند."\n   }\n  },\n  {\n   "id": "n02",\n   "category": "news",\n   "text": "به گفته مدیرعامل این کمپانی، کارخانه تازه مجهز به ربات\u200cهای صنعتی است و تا پایان نوامبر راه\u200cاندازی می\u200cشود.",\n   "marked_in_V": [\n    "گفته",\n    "مدیرعامل",\n    "کمپانی",\n    "مجهز",\n    "ربات\u200cهای",\n    "صنعتی",\n    "است",\n    "نوامبر"\n   ],\n   "app_notes": [\n    "Ezafe written in on 5 word(s) by hazm\'s ParsBERT tagger: گفته، مدیرعامل، کارخانه، ربات\u200cهای، پایان."\n   ],\n   "texts": {\n    "AB": "به گفتهٔ مدیرعاملِ این کمپانی، کارخانهٔ تازه مجهز به ربات\u200cهایِ صنعتی است و تا پایانِ نوامبر راه\u200cاندازی می\u200cشود.",\n    "AV": "به گُفتهٔ مُدیرعاملِ این کُمپانی، کارخانهٔ تازه مُجهز به رُبات\u200cهایِ صَنعتی اَست و تا پایانِ نُوامبر راه\u200cاندازی می\u200cشود."\n   }\n  },\n  {\n   "id": "n03",\n   "category": "news",\n   "text": "تیم ملی فوتبال جمعه شب در ورزشگاه آزادی با نتیجه دو بر یک به حریف عراقی خود باخت.",\n   "marked_in_V": [\n    "ملی",\n    "ورزشگاه",\n    "نتیجه",\n    "حریف",\n    "عراقی"\n   ],\n   "app_notes": [\n    "Ezafe written in on 6 word(s) by hazm\'s ParsBERT tagger: تیم، ملی، ورزشگاه، نتیجه، حریف، عراقی.",\n    "First vowel written on 1 homograph(s) from Homo-GE2PE: بر."\n   ],\n   "texts": {\n    "AB": "تیمِ ملیِ فوتبال جمعه شب در ورزشگاهِ آزادی با نتیجهٔ دو بَر یک به حریفِ عراقیِ خود باخت.",\n    "AV": "تیمِ مِلیِ فوتبال جمعه شب در وَرزشگاهِ آزادی با نَتیجهٔ دو بَر یک به حَریفِ عِراقیِ خود باخت."\n   }\n  },\n  {\n   "id": "n04",\n   "category": "news",\n   "text": "سازمان هواشناسی پیش\u200cبینی کرده است که از سه\u200cشنبه هفته آینده دمای هوا در استان\u200cهای شمالی کاهش می\u200cیابد.",\n   "marked_in_V": [\n    "هواشناسی",\n    "کرده",\n    "است",\n    "سه\u200cشنبه",\n    "هفته",\n    "دمای",\n    "هوا",\n    "استان\u200cهای",\n    "شمالی"\n   ],\n   "app_notes": [\n    "Ezafe written in on 5 word(s) by hazm\'s ParsBERT tagger: سازمان، سه\u200cشنبه، هفته، دمای، استان\u200cهای."\n   ],\n   "texts": {\n    "AB": "سازمانِ هواشناسی پیش\u200cبینی کرده است که از سه\u200cشنبهٔ هفتهٔ آینده دمایِ هوا در استان\u200cهایِ شمالی کاهش می\u200cیابد.",\n    "AV": "سازمانِ هَواشناسی پیش\u200cبینی کَرده اَست که از سِه\u200cشنبهٔ هَفتهٔ آینده دَمایِ هَوا در اُستان\u200cهایِ شُمالی کاهش می\u200cیابد."\n   }\n  },\n  {\n   "id": "n05",\n   "category": "news",\n   "text": "پژوهشگران دانشگاه تهران موفق شدند دارویی برای درمان سرطان خون بسازند که هزینه کمتری دارد.",\n   "marked_in_V": [\n    "پژوهشگران",\n    "تهران",\n    "موفق",\n    "شدند",\n    "برای",\n    "درمان",\n    "سرطان",\n    "بسازند",\n    "هزینه",\n    "کمتری"\n   ],\n   "app_notes": [\n    "Ezafe written in on 6 word(s) by hazm\'s ParsBERT tagger: پژوهشگران، دانشگاه، برای، درمان، سرطان، هزینه."\n   ],\n   "texts": {\n    "AB": "پژوهشگرانِ دانشگاهِ تهران موفق شدند دارویی برایِ درمانِ سرطانِ خون بسازند که هزینهٔ کمتری دارد.",\n    "AV": "پَژوهشگرانِ دانشگاهِ تِهران مُوفق شُدند دارویی بَرایِ دَرمانِ سَرطانِ خون بِسازند که هَزینهٔ کَمتری دارد."\n   }\n  },\n  {\n   "id": "n06",\n   "category": "news",\n   "text": "قیمت دلار در بازار آزاد روز یکشنبه پایین آمد و معامله\u200cگران منتظر تصمیم بانک مرکزی هستند.",\n   "marked_in_V": [\n    "قیمت",\n    "دلار",\n    "یکشنبه",\n    "معامله\u200cگران",\n    "منتظر",\n    "تصمیم",\n    "مرکزی",\n    "هستند"\n   ],\n   "app_notes": [\n    "Ezafe written in on 6 word(s) by hazm\'s ParsBERT tagger: قیمت، بازار، روز، منتظر، تصمیم، بانک."\n   ],\n   "texts": {\n    "AB": "قیمتِ دلار در بازارِ آزاد روزِ یکشنبه پایین آمد و معامله\u200cگران منتظرِ تصمیمِ بانکِ مرکزی هستند.",\n    "AV": "قِیمتِ دُلار در بازارِ آزاد روزِ یِکشنبه پایین آمد و مُعامله\u200cگران مُنتظرِ تَصمیمِ بانکِ مَرکزی هَستند."\n   }\n  },\n  {\n   "id": "n07",\n   "category": "news",\n   "text": "جشنواره فیلم فجر امسال در بهمن ماه برگزار می\u200cشود و هشتاد فیلم کوتاه و بلند در آن حضور دارند.",\n   "marked_in_V": [\n    "جشنواره",\n    "فجر",\n    "امسال",\n    "برگزار",\n    "هشتاد",\n    "بلند",\n    "حضور"\n   ],\n   "app_notes": [\n    "Ezafe written in on 4 word(s) by hazm\'s ParsBERT tagger: جشنواره، فیلم، فجر، فیلم."\n   ],\n   "texts": {\n    "AB": "جشنوارهٔ فیلمِ فجرِ امسال در بهمن ماه برگزار می\u200cشود و هشتاد فیلمِ کوتاه و بلند در آن حضور دارند.",\n    "AV": "جَشنوارهٔ فیلمِ فَجرِ اِمسال در بهمن ماه بَرگزار می\u200cشود و هَشتاد فیلمِ کوتاه و بُلند در آن حُضور دارند."\n   }\n  },\n  {\n   "id": "n08",\n   "category": "news",\n   "text": "وزارت نیرو از مردم خواست در ساعت\u200cهای اوج مصرف، از روشن کردن کولرها و وسایل پرمصرف خودداری کنند.",\n   "marked_in_V": [\n    "وزارت",\n    "مصرف",\n    "کردن",\n    "وسایل",\n    "پرمصرف",\n    "کنند"\n   ],\n   "app_notes": [\n    "Ezafe written in on 5 word(s) by hazm\'s ParsBERT tagger: وزارت، ساعت\u200cهای، اوج، کردن، وسایل.",\n    "First vowel written on 1 homograph(s) from Homo-GE2PE: مردم."\n   ],\n   "texts": {\n    "AB": "وزارتِ نیرو از مَردم خواست در ساعت\u200cهایِ اوجِ مصرف، از روشن کردنِ کولرها و وسایلِ پرمصرف خودداری کنند.",\n    "AV": "وِزارتِ نیرو از مَردم خواست در ساعت\u200cهایِ اوجِ مَصرف، از روشن کَردنِ کولرها و وَسایلِ پُرمصرف خودداری کُنند."\n   }\n  }\n ]\n}\n', encoding="utf-8")
os.environ["VOICECLONE_OMNIVOICE_PYTHON"] = sys.executable
provenance = found[0].parents[2] / "PROVENANCE.md"
os.environ["VOICECLONE_CODE_COMMIT"] = (provenance.read_text().split("commit ")[-1].strip(" .\n")
                                        if provenance.is_file() else "the dataset's PROVENANCE.md was not found")
NEEDED = ['scripts/render_frontend_conditions.py', 'scripts/ab_omnivoice_ipa.py', 'scripts/omnivoice_worker.py', 'src/voiceclone/text/marks_fa.py', 'eval/corpus/manifest.json', 'eval/corpus/mana_000.wav', 'eval/runs/news-vowels-v1/P/texts.json', 'eval/runs/news-vowels-v1/A/texts.json']
missing = [name for name in NEEDED if not (REPO / name).is_file()]
REGISTERED = {'eval/runs/news-vowels-v1/P/texts.json': '8756157ac8b7bed482d08be7ef86a8f2a42f3195712e27ffb87d08df86b258eb', 'eval/runs/news-vowels-v1/A/texts.json': '945fc6a5676e4b54563e84ed93a11b6e5739efeb36218d2c0710cdf93b05d69e'}
for name, digest in REGISTERED.items():
    if (REPO / name).is_file() and hashlib.sha256((REPO / name).read_bytes()).hexdigest() != digest:
        missing.append(f"{name} is not the registered one")
GATES.record("code", not missing, f"missing or changed: {missing}" if missing else
             "the render script, the marks and the registered texts are present; code from " +
             os.environ["VOICECLONE_CODE_COMMIT"])
if not GATES.skip("download", ("runner", "code")):
    fetched_ok, fetched = fetch({'snapshots': [{'name': 'base', 'repo': 'k2-fsa/OmniVoice', 'revision': 'c5fdb5ccb189668d56333f77ba2629f4cd7535f4', 'local_dir': '/tmp/omnivoice-base', 'sha256': {'audio_tokenizer/model.safetensors': 'fe7c5e8785e0a05833e1bfc3e002ec7f55af21e306b2e7154a448c1f54ccfb0d'}}]}, 20, ['--no-xet'])
    GATES.record("download", fetched_ok, json.dumps(fetched))
# The engine reads its weights from this override; nothing else differs from the Mac's settings.
os.environ["VOICECLONE__ENGINES__OMNIVOICE__MODEL_ID"] = "/tmp/omnivoice-base"

MODELS = {'P': None, 'A': 'export-1000-float16'}
KNOWN_SHA256 = {'A': '9c453a74e9eab8324c27e039b9bdc052cd8d6721cecbb20df53c919a0e6ba39a'}
WEIGHTS = {}
if not GATES.skip("weights", ("download",)):
    problems = []
    for form, name in MODELS.items():
        if name is None:
            WEIGHTS[form] = {"path": "/tmp/omnivoice-base"}
            continue
        hits = sorted(p for p in INPUT.rglob(name) if p.is_dir() and (p / "model.safetensors").is_file())
        if not hits:
            problems.append(f"{form}: no {name} with a model.safetensors under /kaggle/input")
            continue
        digest = hashlib.sha256()
        with open(hits[0] / "model.safetensors", "rb") as handle:
            for block in iter(lambda: handle.read(1 << 24), b""):
                digest.update(block)
        WEIGHTS[form] = {"path": str(hits[0]), "found": [str(p) for p in hits], "model_sha256": digest.hexdigest()}
        if form in KNOWN_SHA256 and digest.hexdigest() != KNOWN_SHA256[form]:
            problems.append(f"{form}: model.safetensors is not the one recorded before")
    GATES.record("weights", not problems, json.dumps({"weights": WEIGHTS, "problems": problems}))


In [ ]:
OUT = REPO / "eval/runs/news-vowels-v1"
RESULT = {}
for form in MODELS:
    gate = f"renders-{form}"
    if GATES.skip(gate, ("weights",)):
        continue
    os.environ["VOICECLONE__ENGINES__OMNIVOICE__MODEL_ID"] = WEIGHTS[form]["path"]
    run = run_bounded([sys.executable, str(REPO / "scripts/render_frontend_conditions.py"), "--cer", "--texts",
                       str(OUT / form / "texts.json"), "--app-normaliser"], cwd=REPO,
                      timeout=60 * 60, log=LOGS / f"render-{form}.log", monitor_every=300)
    RUNS[gate] = run.record()
    print(run.summary())
    manifest_path = OUT / form / "manifest.json"
    manifest = json.loads(manifest_path.read_text()) if manifest_path.is_file() else {"renders": [], "events": []}
    rendered = [r for r in manifest["renders"] if r.get("wav") and Path(r["wav"]).is_file()]
    scored = [r for r in rendered if r.get("cer") is not None]
    text = run.log.read_text(errors="replace") if run.log.is_file() else ""
    checked = [e for e in manifest["events"] if e.get("event") == "marks_checked_through_the_text_pipeline"]
    used = manifest.get("settings", {}).get("omnivoice_model_id")
    RESULT[form] = {"rendered": len(rendered), "scored": len(scored), "weights": used,
                    "worker": [line.strip() for line in text.splitlines() if "OmniVoice ready" in line][:1],
                    "mean_cer": round(sum(r["cer"] for r in scored) / len(scored), 4) if scored else None,
                    "other_changes_by_the_app": checked[0]["other_changes_by_the_app"] if checked else None}
    GATES.record(gate, run.ok and len(rendered) == 16 and len(scored) == 16 and bool(checked)
                 and used == WEIGHTS[form]["path"], json.dumps(RESULT[form], ensure_ascii=False))
os.environ["VOICECLONE__ENGINES__OMNIVOICE__MODEL_ID"] = "/tmp/omnivoice-base"


In [ ]:
destination = Path("/kaggle/working/news-vowels-v1")
destination.mkdir(exist_ok=True)
for form in MODELS:
    (destination / form).mkdir(exist_ok=True)
    for name in ("manifest.json", "texts.json"):
        if (OUT / form / name).is_file():
            shutil.copy2(OUT / form / name, destination / form / name)
    if (OUT / form / "wavs").is_dir():
        shutil.copytree(OUT / form / "wavs", destination / form / "wavs", dirs_exist_ok=True)
(destination / "gates.json").write_text(json.dumps(GATES.summary(), indent=2))
(destination / "runs.json").write_text(json.dumps(RUNS, indent=2))
(destination / "result.json").write_text(json.dumps({"renders": RESULT, "weights": WEIGHTS}, indent=2,
                                                    ensure_ascii=False))
shutil.copy2("/kaggle/working/environment.json", destination / "environment.json")
print("gates:", json.dumps(GATES.summary(), indent=1))
